# B5: Quotas and rate limits per team/application
The chatbot team's MaaSSubscription token limit is lowered live (100k → 3,000 tokens/min) by the platform admin
(`scripts/b5-set-limit.sh 3000`). Then we exceed it: requests above the limit get **HTTP 429**.

In [ ]:
import os, json, time, urllib.request, ssl
from concurrent.futures import ThreadPoolExecutor
URL = os.environ["MAAS_URL"] + "/v1/chat/completions"; KEY = os.environ["KEY_CHATBOT"]
def call(_):
    body = json.dumps({"model":os.environ["MODEL_ID"],"max_tokens":300,
                       "messages":[{"role":"user","content":"Write 150 words about Islamic microfinance."}]}).encode()
    req = urllib.request.Request(URL, data=body, headers={"Authorization":"Bearer "+KEY,"Content-Type":"application/json"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r: return r.status, json.load(r)["usage"]["total_tokens"]
    except urllib.error.HTTPError as e: return e.code, 0

In [ ]:
t0 = time.time(); results = []
with ThreadPoolExecutor(8) as ex:
    for _ in range(3): results += list(ex.map(call, range(8)))
from collections import Counter
print(Counter(c for c,_ in results), "| tokens served:", sum(t for _,t in results), f"| {time.time()-t0:.0f}s")

**Reading the output:** `Counter({200: X, 429: Y})` counts responses by HTTP status.

* `200`: request served within the team's limit.
* `429`: rejected by the gateway because the team exceeded its token limit (Too Many Requests). Expect about **16 × 200 and 8 × 429** with the limit at 3,000 tokens/min.
* `tokens served`: total tokens actually delivered. It's a bit above 3,000 because tokens are counted **after** each response and requests already in flight are allowed to finish (about 25% overshoot).
* The last number is how long the 24 requests took.

No 429 at all usually means the limit isn't lowered yet (run `scripts/b5-set-limit.sh 3000` first) or a new minute window just started; run the cell again. **Afterwards, restore the limit** with `scripts/b5-set-limit.sh 100000`.